# 01 · Exploratory analysis of cleaned data and labelled matches

Goal: understand what a *true match* looks like so the blocking key and features are chosen from
evidence. Inputs: cleaned Parquet from notebook 00 (only the needed columns are loaded).

In [ ]:
import sys, time, gc
from pathlib import Path

# notebooks/ -> ../src holds the shared, importable pipeline package `ber`
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ber import config as C, io, pipeline as P

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
C.ensure_dirs()
print(C.describe())
try:
    import psutil
    print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB | CPUs: {psutil.cpu_count()}")
except ImportError:
    pass

In [ ]:
cols = ["entity_id", "country", "name_core", "addr_clean", "addr_nums", "name_nonlatin", "name_ntok", "addr_ntok"]
s1 = pd.read_parquet(io.fetch("clean/train_s1.parquet"), columns=cols)
r = pd.concat([pd.read_parquet(io.fetch("clean/train_s2.parquet"), columns=cols),
               pd.read_parquet(io.fetch("clean/train_s3.parquet"), columns=cols)], ignore_index=True)
gt = io.load_parquet("clean/train_gt.parquet")
tp = P.load_true_pairs()
print(f"S1 {len(s1):,} | R (S2+S3) {len(r):,} | GT rows {len(gt):,} | true pairs {len(tp):,}")

## 1. Length / token distributions after cleaning

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 6))
for j, (k, d) in enumerate((("S1", s1), ("S2+S3", r))):
    d["name_ntok"].clip(upper=10).value_counts().sort_index().plot.bar(ax=axes[0, j], title=f"{k}: name_core tokens")
    d["addr_ntok"].clip(upper=25).value_counts().sort_index().plot.bar(ax=axes[1, j], title=f"{k}: addr_clean tokens")
plt.tight_layout(); plt.show()

## 2. Duplicates within sources

In [ ]:
for k, d in (("S1", s1), ("S2+S3", r)):
    print(f"{k}: same cleaned name {100 * d.duplicated(['country', 'name_core']).mean():.2f}% | "
          f"same name+address {100 * d.duplicated(['country', 'name_core', 'addr_clean']).mean():.2f}%")

S1 is deduplicated as entities, yet many S1 records share a cleaned name (`family health`,
`grace chapel`): **the name alone cannot identify an entity** — the address must take part in
both retrieval and scoring.

## 3. Ground-truth structure

In [ ]:
n_match = gt["matched_entity_ids"].fillna("").map(lambda s: 0 if not s else s.count(",") + 1)
print(f"singletons (no match): {100 * (n_match == 0).mean():.2f}% of S1")
n_match.value_counts().sort_index().plot.bar(figsize=(8, 3), title="matches per S1 entity"); plt.show()
print("matched records by source:", tp["candidate_entity_id"].str[:2].value_counts().to_dict())
print(f"S2/S3 records matched to MORE THAN ONE S1: {tp['candidate_entity_id'].duplicated().sum()}")
print(f"share of all S2/S3 records that have a match: {100 * len(tp) / len(r):.1f}%")

**Key structural fact:** every S2/S3 record belongs to **at most one** S1 entity. So retrieval
searches from each S2/S3 record for its single best S1 (a small top-K suffices), and features can
measure how clearly one S1 beats its rivals for the same record.

## 4. What do true matches look like?

In [ ]:
from rapidfuzz import fuzz
from rapidfuzz.process import cpdist
smp = tp.sample(min(200_000, len(tp)), random_state=0)
A = s1.set_index("entity_id").loc[smp["source1_entity_id"]].reset_index()
B = r.set_index("entity_id").loc[smp["candidate_entity_id"]].reset_index()
name_sim = cpdist(A.name_core.tolist(), B.name_core.tolist(), scorer=fuzz.token_set_ratio, workers=C.N_JOBS)
addr_sim = cpdist(A.addr_clean.tolist(), B.addr_clean.tolist(), scorer=fuzz.token_set_ratio, workers=C.N_JOBS)
num_shared = np.array([bool(set(x.split()) & set(y.split())) for x, y in zip(A.addr_nums, B.addr_nums)])
print(f"country agrees          : {100 * (A.country.values == B.country.values).mean():.3f}%")
print(f"exact same name_core    : {100 * (A.name_core.values == B.name_core.values).mean():.1f}%")
print(f"share >=1 address number: {100 * num_shared.mean():.1f}%")
print(f"S2/S3 address empty     : {100 * (B.addr_clean == '').mean():.1f}%")
print(f"S2/S3 name non-Latin    : {100 * B.name_nonlatin.mean():.1f}%")
cat = pd.cut(name_sim, [-1, 50, 80, 99.9, 100], labels=["different name (<50)", "paraphrase (50-80)", "near-dup (80-99)", "exact (100)"])
acat = pd.cut(addr_sim, [-1, 50, 80, 100], labels=["addr <50", "addr 50-80", "addr 80-100"])
pd.crosstab(cat, acat, normalize=True).round(3)

In [ ]:
def show(mask, n=6, title=""):
    print(f"--- {title} ---")
    for i in np.flatnonzero(mask)[:n]:
        print(f"  S1: {A.name_core[i]!r:40} | {A.addr_clean[i]!r}")
        print(f"  {B.entity_id[i][:2]}: {B.name_core[i]!r:40} | {B.addr_clean[i]!r}\n")
show(name_sim == 100, title="exact name")
show((name_sim >= 50) & (name_sim < 80), title="paraphrase / noisy name")
show(name_sim < 50, title="different name - must be matched by address")

## 5. Conclusions for blocking and features

- **Country agrees in 100% of true pairs** → lossless hard block (France forms its own block).
- Many true pairs have **very different names** (renamed entities, DBA names, transliterations)
  but share address tokens/numbers → retrieval must use the **address as well as the name**.
- Names are typo-heavy → **character n-grams**; addresses are reordered token bags → **word n-grams**.
- Many S1 records share a generic name → scoring must be **relative**: how much better this S1 is
  than the other candidates for the same record ("competition" features).

In [ ]:
del s1, r, A, B; gc.collect()